In [15]:
import os
from langchain_aws import BedrockEmbeddings, ChatBedrock

def get_models():

    embedding_model = BedrockEmbeddings(
        model_id="amazon.titan-embed-text-v1",
        region_name="us-east-1"
    )

    chat_model = ChatBedrock(
        model_id = "anthropic.claude-3-haiku-20240307-v1:0",
        region_name = "us-east-1", 
        model_kwargs={
            "temperature": 0.1
        }
    )

    return embedding_model, chat_model

In [16]:
!pip install ragas


In [17]:
from datasets import Dataset

embedding_model, chat_model = get_models()

data_samples = {
    "question": [
        "What is the company vacation policy?",
        "What is the company vacation policy?"
    ],
    "answer":[
        "According to the employee handbook, you get 15 days of PTO each year.",
        "You get no PTO. Vacations and sick leave are not allowed at all."
    ],
    "contexts":[
        ["Employees reieve 15 days of PTO per calendar year. You may carry 40 hours of PTO forward to the next year."],
        ["Employees reieve 15 days of PTO per calendar year. You may carry 40 hours of PTO forward to the next year."]
    ]
}

eval_dataset = Dataset.from_dict(data_samples)
print(eval_dataset)


Dataset({
    features: ['question', 'answer', 'contexts'],
    num_rows: 2
})


In [19]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
import pandas as pd

metrics = [faithfulness, answer_relevancy]

response = evaluate(
    dataset = eval_dataset,
    metrics = metrics,
    llm = chat_model,
    embeddings = embedding_model
)

print(response)

df = response.to_pandas()

C:\Users\RichardHawkins\AppData\Local\Temp\ipykernel_4884\2288822232.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
C:\Users\RichardHawkins\AppData\Local\Temp\ipykernel_4884\2288822232.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy
Evaluating: 100%|██████████| 4/4 [00:11<00:00,  2.76s/it]


{'faithfulness': 0.5000, 'answer_relevancy': 0.6534}
